In [1]:
from pathlib import Path
import sys

import ee
import pandas as pd


# ============================================================
# Repository setup
# ============================================================

repo_root = Path.cwd().resolve()

if repo_root.name == "diagnostics":
    repo_root = repo_root.parent.parent
elif repo_root.name == "notebooks":
    repo_root = repo_root.parent

src_path = repo_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


# ============================================================
# Earth Engine
# ============================================================

ee.Initialize(project="ee-change")


# ============================================================
# Project imports
# ============================================================

from et_downscaling.modis import build_modis_inputs

import et_downscaling.hls as hls

from et_downscaling.hls import (
    get_hls_collection,
    build_hls_predictors,
)


print("Repository:", repo_root)
print("HLS module:", hls.__file__)


# ============================================================
# Station footprints
# ============================================================

modis_inputs = build_modis_inputs()

station_footprints = modis_inputs[
    "station_footprints"
]

qa_geometry = station_footprints.geometry()

print(
    "Stations:",
    station_footprints.size().getInfo(),
)


# ============================================================
# QA periods
# ============================================================

qa_windows = [
    ("2021-01-01", "2021-01-17"),
    ("2021-04-01", "2021-04-17"),
    ("2021-07-01", "2021-07-17"),
    ("2021-10-01", "2021-10-17"),
    ("2022-01-01", "2022-01-17"),
    ("2022-04-01", "2022-04-17"),
    ("2022-07-01", "2022-07-17"),
    ("2022-10-01", "2022-10-17"),
    ("2023-01-01", "2023-01-17"),
    ("2023-04-01", "2023-04-17"),
    ("2023-07-01", "2023-07-17"),
    ("2023-10-01", "2023-10-17"),
]


# ============================================================
# HLS collection
# ============================================================

hls_collection = get_hls_collection(
    station_footprints
)


# ============================================================
# Statistics reducer
# ============================================================

stats_reducer = (
    ee.Reducer.minMax()
    .combine(
        reducer2=ee.Reducer.percentile(
            [1, 5, 50, 95, 99]
        ),
        sharedInputs=True,
    )
    .combine(
        reducer2=ee.Reducer.mean(),
        sharedInputs=True,
    )
    .combine(
        reducer2=ee.Reducer.count(),
        sharedInputs=True,
    )
)


# ============================================================
# Run HLS QA
# ============================================================

rows = []

for start_date, end_date in qa_windows:

    period_collection = (
        hls_collection
        .filterDate(
            start_date,
            end_date,
        )
    )

    predictors = build_hls_predictors(
        period_collection,
        qa_geometry,
    )

    bands = predictors.bandNames().getInfo()

    negative = (
        predictors
        .lt(0)
        .rename([
            f"{band}__negative"
            for band in bands
        ])
    )

    above_one = (
        predictors
        .gt(1)
        .rename([
            f"{band}__above_one"
            for band in bands
        ])
    )

    below_minus_one = (
        predictors
        .lt(-1)
        .rename([
            f"{band}__below_minus_one"
            for band in bands
        ])
    )

    abs_above_two = (
        predictors
        .abs()
        .gt(2)
        .rename([
            f"{band}__abs_above_two"
            for band in bands
        ])
    )

    diagnostic_flags = (
        negative
        .addBands(above_one)
        .addBands(below_minus_one)
        .addBands(abs_above_two)
    )

    result = ee.Dictionary({
        "product_count": period_collection.size(),

        "stats": predictors.reduceRegion(
            reducer=stats_reducer,
            geometry=qa_geometry,
            scale=30,
            maxPixels=1000000,
        ),

        "flags": diagnostic_flags.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=qa_geometry,
            scale=30,
            maxPixels=1000000,
        ),
    }).getInfo()

    stats = result["stats"]
    flags = result["flags"]

    print(
        start_date,
        "products:",
        result["product_count"],
    )

    for band in bands:

        negative_value = flags.get(
            f"{band}__negative"
        )

        above_one_value = flags.get(
            f"{band}__above_one"
        )

        below_minus_one_value = flags.get(
            f"{band}__below_minus_one"
        )

        abs_above_two_value = flags.get(
            f"{band}__abs_above_two"
        )

        rows.append({
            "period": start_date,
            "band": band,

            "min": stats.get(
                f"{band}_min"
            ),

            "p01": stats.get(
                f"{band}_p1"
            ),

            "p05": stats.get(
                f"{band}_p5"
            ),

            "median": stats.get(
                f"{band}_p50"
            ),

            "mean": stats.get(
                f"{band}_mean"
            ),

            "p95": stats.get(
                f"{band}_p95"
            ),

            "p99": stats.get(
                f"{band}_p99"
            ),

            "max": stats.get(
                f"{band}_max"
            ),

            "negative_pct": (
                negative_value * 100
                if negative_value is not None
                else None
            ),

            "above_one_pct": (
                above_one_value * 100
                if above_one_value is not None
                else None
            ),

            "below_minus_one_pct": (
                below_minus_one_value * 100
                if below_minus_one_value is not None
                else None
            ),

            "abs_above_two_pct": (
                abs_above_two_value * 100
                if abs_above_two_value is not None
                else None
            ),

            "n": stats.get(
                f"{band}_count"
            ),
        })


# ============================================================
# QA summary
# ============================================================

qa_full = pd.DataFrame(rows)

qa_summary = (
    qa_full
    .groupby(
        "band",
        as_index=False,
    )
    .agg(
        periods=("period", "count"),
        min=("min", "min"),
        lowest_p01=("p01", "min"),
        median=("median", "median"),
        highest_p99=("p99", "max"),
        max=("max", "max"),
        max_negative_pct=(
            "negative_pct",
            "max",
        ),
        max_above_one_pct=(
            "above_one_pct",
            "max",
        ),
        max_below_minus_one_pct=(
            "below_minus_one_pct",
            "max",
        ),
        max_abs_above_two_pct=(
            "abs_above_two_pct",
            "max",
        ),
        total_n=("n", "sum"),
    )
)


print("\nHLS FINAL QA\n")

print(
    qa_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)

c:\Users\usrlabsis22\AppData\Local\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


Repository: E:\1SIG22\Varios\github\git_github\26-et-downscaling-fundacion
HLS module: E:\1SIG22\Varios\github\git_github\26-et-downscaling-fundacion\src\et_downscaling\hls.py


c:\Users\usrlabsis22\AppData\Local\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


Stations: 5
2021-01-01 products: 14
2021-04-01 products: 12
2021-07-01 products: 10
2021-10-01 products: 14
2022-01-01 products: 15
2022-04-01 products: 18
2022-07-01 products: 103
2022-10-01 products: 109
2023-01-01 products: 33
2023-04-01 products: 117
2023-07-01 products: 105
2023-10-01 products: 96

HLS FINAL QA

   band  periods     min  lowest_p01  median  highest_p99    max  max_negative_pct  max_above_one_pct  max_below_minus_one_pct  max_abs_above_two_pct  total_n
   Blue       12  0.0003      0.0031  0.0224       0.3046 0.3761            0.0000                  0                        0                      0    13034
Coastal       12 -0.0499     -0.0124  0.0191       0.2764 0.3538            7.3913                  0                        0                      0    13034
    EVI       12 -0.0731     -0.0449  0.5936       0.8380 0.9005            4.1975                  0                        0                      0    13034
  Green       12  0.0187      0.0249  0.0590 

In [2]:
from pathlib import Path
import sys

import ee
import pandas as pd


repo_root = Path.cwd().resolve()

if repo_root.name == "diagnostics":
    repo_root = repo_root.parent.parent
elif repo_root.name == "notebooks":
    repo_root = repo_root.parent

src_path = repo_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


ee.Initialize(project="ee-change")


from et_downscaling.modis import build_modis_inputs


modis_inputs = build_modis_inputs()

modis_collection = modis_inputs["collection"]
modis_projection = modis_inputs["projection"]
modis_scale = modis_inputs["scale"]
station_footprints = modis_inputs["station_footprints"]


modis_list = modis_collection.toList(
    modis_collection.size()
)

modis_indices = ee.List.sequence(
    0,
    modis_collection.size().subtract(1),
)

footprint_list = station_footprints.toList(
    station_footprints.size()
)

footprint_indices = ee.List.sequence(
    0,
    station_footprints.size().subtract(1),
)


def process_modis_image(modis_index):
    image = ee.Image(
        modis_list.get(modis_index)
    )

    period_start = image.date()

    def process_station(station_index):
        footprint = ee.Feature(
            footprint_list.get(station_index)
        )

        station_point = ee.Geometry.Point(
            [
                footprint.get("longitude"),
                footprint.get("latitude"),
            ]
        )

        values = image.reduceRegion(
            reducer=ee.Reducer.first(),
            geometry=station_point,
            crs=modis_projection,
            scale=modis_scale,
            maxPixels=10,
        )

        et_raw = values.get("ET")
        qc_raw = values.get("ET_QC")

        et_present = ee.Number(
            ee.Algorithms.If(
                ee.Algorithms.IsEqual(
                    et_raw,
                    None,
                ),
                0,
                1,
            )
        )

        qc_present = ee.Number(
            ee.Algorithms.If(
                ee.Algorithms.IsEqual(
                    qc_raw,
                    None,
                ),
                0,
                1,
            )
        )

        et_dn = ee.Number(
            ee.Algorithms.If(
                et_present.eq(1),
                et_raw,
                -99999,
            )
        )

        qc = ee.Number(
            ee.Algorithms.If(
                qc_present.eq(1),
                qc_raw,
                255,
            )
        ).toInt()

        modland_qc = (
            qc
            .bitwiseAnd(1)
        )

        detector_qc = (
            qc
            .rightShift(2)
            .bitwiseAnd(1)
        )

        cloud_state = (
            qc
            .rightShift(3)
            .bitwiseAnd(3)
        )

        scf_qc = (
            qc
            .rightShift(5)
            .bitwiseAnd(7)
        )

        official_value_valid = (
            et_present
            .multiply(
                et_dn.gte(-32767)
            )
            .multiply(
                et_dn.lte(32700)
            )
            .toInt()
        )

        nonnegative_value_valid = (
            et_present
            .multiply(
                et_dn.gte(0)
            )
            .multiply(
                et_dn.lte(32700)
            )
            .toInt()
        )

        current_strict = (
            qc_present
            .multiply(
                nonnegative_value_valid
            )
            .multiply(
                modland_qc.eq(0)
            )
            .multiply(
                detector_qc.eq(0)
            )
            .multiply(
                scf_qc.lte(1)
            )
            .toInt()
        )

        recovered_qc_only = (
            nonnegative_value_valid
            .multiply(
                ee.Number(1)
                .subtract(
                    current_strict
                )
            )
            .toInt()
        )

        negative_official_value = (
            official_value_valid
            .multiply(
                et_dn.lt(0)
            )
            .toInt()
        )

        return ee.Feature(
            None,
            {
                "station":
                    footprint.get("station"),

                "station_id":
                    footprint.get("station_id"),

                "period_start":
                    period_start.format(
                        "yyyy-MM-dd"
                    ),

                "year":
                    period_start.get("year"),

                "ET_DN":
                    et_dn,

                "ET_mm_period":
                    et_dn.multiply(0.1),

                "ET_QC":
                    qc,

                "modland_qc":
                    modland_qc,

                "detector_qc":
                    detector_qc,

                "cloud_state":
                    cloud_state,

                "scf_qc":
                    scf_qc,

                "official_value_valid":
                    official_value_valid,

                "nonnegative_value_valid":
                    nonnegative_value_valid,

                "current_strict":
                    current_strict,

                "recovered_qc_only":
                    recovered_qc_only,

                "negative_official_value":
                    negative_official_value,
            },
        )

    return footprint_indices.map(
        process_station
    )


diagnostic_table = ee.FeatureCollection(
    ee.List(
        modis_indices.map(
            process_modis_image
        )
    ).flatten()
)


records = [
    feature["properties"]
    for feature in diagnostic_table
    .getInfo()["features"]
]

df = pd.DataFrame(records)


print("\nMOD16A2GF QC DIAGNOSTIC\n")

print(
    "Station-period observations:",
    len(df),
)

print(
    "Current strict:",
    int(df["current_strict"].sum()),
)

print(
    "Valid ET, QC ignored, ET >= 0:",
    int(df["nonnegative_value_valid"].sum()),
)

print(
    "Official product-valid range:",
    int(df["official_value_valid"].sum()),
)

print(
    "Recovered only by removing QC:",
    int(df["recovered_qc_only"].sum()),
)

print(
    "Official-valid negative ET:",
    int(df["negative_official_value"].sum()),
)


recovered = df[
    df["recovered_qc_only"] == 1
].copy()


if len(recovered) > 0:

    print("\nRECOVERED ET DISTRIBUTION\n")

    print(
        recovered["ET_mm_period"]
        .describe(
            percentiles=[
                0.01,
                0.05,
                0.50,
                0.95,
                0.99,
            ]
        )
        .to_string()
    )

    print("\nRECOVERED BY STATION\n")

    print(
        recovered
        .groupby("station")
        .size()
        .to_string()
    )

    print("\nRECOVERED BY YEAR\n")

    print(
        recovered
        .groupby("year")
        .size()
        .to_string()
    )

    print("\nQC CHARACTERISTICS OF RECOVERED VALUES\n")

    print(
        "MODLAND_QC != 0:",
        int(
            (
                recovered["modland_qc"] != 0
            ).sum()
        ),
    )

    print(
        "Detector flag != 0:",
        int(
            (
                recovered["detector_qc"] != 0
            ).sum()
        ),
    )

    print(
        "SCF_QC > 1:",
        int(
            (
                recovered["scf_qc"] > 1
            ).sum()
        ),
    )

    print("\nSCF_QC DISTRIBUTION\n")

    print(
        recovered["scf_qc"]
        .value_counts()
        .sort_index()
        .to_string()
    )


negative_et = df[
    df["negative_official_value"] == 1
].copy()


if len(negative_et) > 0:

    print("\nOFFICIAL-VALID NEGATIVE ET\n")

    print(
        negative_et[
            [
                "station",
                "period_start",
                "ET_DN",
                "ET_mm_period",
                "ET_QC",
                "scf_qc",
            ]
        ]
        .sort_values(
            "ET_mm_period"
        )
        .to_string(
            index=False
        )
    )


MOD16A2GF QC DIAGNOSTIC

Station-period observations: 690
Current strict: 610
Valid ET, QC ignored, ET >= 0: 690
Official product-valid range: 690
Recovered only by removing QC: 80
Official-valid negative ET: 0

RECOVERED ET DISTRIBUTION

count    80.000000
mean     33.421250
std       8.245531
min      17.900000
1%       18.769000
5%       22.065000
50%      31.950000
95%      47.715000
99%      57.050000
max      61.000000

RECOVERED BY STATION

station
Bananera          24
Bosque seco       14
Manglar           15
Palma             23
Pastos limpios     4

RECOVERED BY YEAR

year
2021    27
2022    36
2023    17

QC CHARACTERISTICS OF RECOVERED VALUES

MODLAND_QC != 0: 80
Detector flag != 0: 5
SCF_QC > 1: 80

SCF_QC DISTRIBUTION

scf_qc
3    75
7     5


c:\Users\usrlabsis22\AppData\Local\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


In [4]:
from pathlib import Path
import sys
import ee

repo_root = Path.cwd().resolve()

if repo_root.name == "diagnostics":
    repo_root = repo_root.parent.parent
elif repo_root.name == "notebooks":
    repo_root = repo_root.parent

src_path = repo_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

ee.Initialize(project="ee-change")

from et_downscaling.config import MODIS_REQUIRE_STRICT_QC
from et_downscaling.modis import (
    build_modis_inputs,
    get_modis_period_end,
    prepare_modis_et,
)

modis_inputs = build_modis_inputs()

modis_collection = modis_inputs["collection"]
station_footprints = modis_inputs["station_footprints"]

image_list = modis_collection.toList(
    modis_collection.size()
)

image_indices = ee.List.sequence(
    0,
    modis_collection.size().subtract(1),
)


def extract_period(index):
    image = ee.Image(
        image_list.get(index)
    )

    period_start = image.date()

    period_end = get_modis_period_end(
        period_start
    )

    number_days = (
        period_end
        .difference(
            period_start,
            "day",
        )
    )

    prepared = prepare_modis_et(
        image,
        number_days,
    )

    values = prepared.select(
        [
            "modis_value_valid",
            "modis_qc_good",
            "modis_good",
        ]
    ).reduceRegions(
        collection=station_footprints,
        reducer=ee.Reducer.first(),
        scale=500,
    )

    return values


observations = ee.FeatureCollection(
    image_indices
    .map(extract_period)
    .flatten()
)

print(
    "MODIS_REQUIRE_STRICT_QC:",
    MODIS_REQUIRE_STRICT_QC,
)

print(
    "Station-period observations:",
    observations.size().getInfo(),
)

print(
    "modis_value_valid:",
    observations.aggregate_sum(
        "modis_value_valid"
    ).getInfo(),
)

print(
    "modis_qc_good:",
    observations.aggregate_sum(
        "modis_qc_good"
    ).getInfo(),
)

print(
    "modis_good:",
    observations.aggregate_sum(
        "modis_good"
    ).getInfo(),
)

ImportError: cannot import name 'MODIS_REQUIRE_STRICT_QC' from 'et_downscaling.config' (E:\1SIG22\Varios\github\git_github\26-et-downscaling-fundacion\src\et_downscaling\config.py)

In [1]:
from pathlib import Path
import sys
import ee

repo_root = Path.cwd().resolve()

if repo_root.name == "diagnostics":
    repo_root = repo_root.parent.parent
elif repo_root.name == "notebooks":
    repo_root = repo_root.parent

src_path = repo_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

ee.Initialize(project="ee-change")

from et_downscaling.config import MODIS_REQUIRE_STRICT_QC

print("MODIS_REQUIRE_STRICT_QC:", MODIS_REQUIRE_STRICT_QC)

c:\Users\usrlabsis22\AppData\Local\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


MODIS_REQUIRE_STRICT_QC: False


In [3]:
from pathlib import Path
import sys
import ee

repo_root = Path.cwd().resolve()

if repo_root.name == "diagnostics":
    repo_root = repo_root.parent.parent
elif repo_root.name == "notebooks":
    repo_root = repo_root.parent

src_path = repo_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

ee.Initialize(project="ee-change")

from et_downscaling.config import MODIS_REQUIRE_STRICT_QC
from et_downscaling.modis import (
    build_modis_inputs,
    get_modis_period_end,
    prepare_modis_et,
)

modis_inputs = build_modis_inputs()

modis_collection = modis_inputs["collection"]
station_footprints = modis_inputs["station_footprints"]

image = ee.Image(
    modis_collection.first()
)

period_start = image.date()

period_end = get_modis_period_end(
    period_start
)

number_days = period_end.difference(
    period_start,
    "day",
)

prepared = prepare_modis_et(
    image,
    number_days,
)

print(
    "MODIS_REQUIRE_STRICT_QC:",
    MODIS_REQUIRE_STRICT_QC,
)

print(
    "Bands:",
    prepared.bandNames().getInfo(),
)

sample = (
    prepared
    .select(
        [
            "ET_mm_period",
            "modis_value_valid",
            "modis_qc_good",
            "modis_good",
            "modis_scf_qc",
        ]
    )
    .sampleRegions(
        collection=station_footprints,
        scale=500,
        geometries=False,
    )
)

print(
    "Stations sampled:",
    sample.size().getInfo(),
)

print(
    "Value valid:",
    sample.aggregate_sum(
        "modis_value_valid"
    ).getInfo(),
)

print(
    "QC good:",
    sample.aggregate_sum(
        "modis_qc_good"
    ).getInfo(),
)

print(
    "Active modis_good:",
    sample.aggregate_sum(
        "modis_good"
    ).getInfo(),
)

difference = (
    prepared
    .select("modis_good")
    .neq(
        prepared.select(
            "modis_value_valid"
        )
    )
)

print(
    "modis_good differs from value_valid:",
    difference.reduceRegion(
        reducer=ee.Reducer.max(),
        geometry=station_footprints.geometry(),
        scale=500,
        maxPixels=1000,
    ).getInfo(),
)

MODIS_REQUIRE_STRICT_QC: False
Bands: ['ET_mm_period', 'ET_mm_day', 'modis_value_valid', 'modis_good', 'ET_QC', 'modis_qc_present', 'modis_qc_good', 'modis_modland_qc', 'modis_sensor', 'modis_dead_detector', 'modis_cloud_state', 'modis_scf_qc']
Stations sampled: 5
Value valid: 5
QC good: 5
Active modis_good: 5
modis_good differs from value_valid: {'modis_good': 0}


In [4]:
from pathlib import Path
import sys

import ee
import pandas as pd


repo_root = Path.cwd().resolve()

if repo_root.name == "diagnostics":
    repo_root = repo_root.parent.parent
elif repo_root.name == "notebooks":
    repo_root = repo_root.parent

src_path = repo_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

ee.Initialize(project="ee-change")


from et_downscaling.modis import (
    build_modis_inputs,
    get_modis_period_end,
)

from et_downscaling.sentinel2 import (
    get_sentinel2_collection,
    build_s2_medoid,
)

from et_downscaling.hls import (
    get_hls_collection,
    build_hls_medoid,
)


modis_inputs = build_modis_inputs()

modis_collection = modis_inputs["collection"]
station_footprints = modis_inputs["station_footprints"]

s2_collection = get_sentinel2_collection(
    station_footprints
)

hls_collection = get_hls_collection(
    station_footprints
)


footprints = (
    station_footprints
    .getInfo()["features"]
)

modis_dates = (
    modis_collection
    .aggregate_array("system:time_start")
    .getInfo()
)


rows = []


def calculate_coverage(
    image,
    geometry,
    scale,
):
    valid = (
        image
        .select("Blue")
        .mask()
        .gt(0)
    )

    valid_count = (
        valid
        .reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=geometry,
            scale=scale,
            maxPixels=100000,
        )
        .get("Blue")
        .getInfo()
    )

    total_count = (
        ee.Image.constant(1)
        .reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=geometry,
            scale=scale,
            maxPixels=100000,
        )
        .get("constant")
        .getInfo()
    )

    valid_count = valid_count or 0
    total_count = total_count or 0

    coverage = (
        100 * valid_count / total_count
        if total_count > 0
        else None
    )

    return valid_count, total_count, coverage


for timestamp in modis_dates:

    start = ee.Date(timestamp)
    end = get_modis_period_end(start)

    start_string = start.format(
        "yyyy-MM-dd"
    ).getInfo()

    end_string = end.format(
        "yyyy-MM-dd"
    ).getInfo()

    s2_period = s2_collection.filterDate(
        start,
        end,
    )

    hls_period = hls_collection.filterDate(
        start,
        end,
    )

    s2_products = s2_period.size().getInfo()
    hls_products = hls_period.size().getInfo()

    for footprint_info in footprints:

        properties = footprint_info["properties"]

        station = properties["station"]

        geometry = ee.Feature(
            footprint_info
        ).geometry()

        if s2_products > 0:

            s2_medoid = build_s2_medoid(
                s2_period,
                geometry,
            )

            valid, total, coverage = (
                calculate_coverage(
                    s2_medoid,
                    geometry,
                    20,
                )
            )

            rows.append({
                "source": "S2",
                "station": station,
                "period_start": start_string,
                "period_end": end_string,
                "products": s2_products,
                "valid_pixels": valid,
                "total_pixels": total,
                "coverage_pct": coverage,
            })

        if hls_products > 0:

            hls_medoid = build_hls_medoid(
                hls_period,
                geometry,
            )

            valid, total, coverage = (
                calculate_coverage(
                    hls_medoid,
                    geometry,
                    30,
                )
            )

            rows.append({
                "source": "HLS",
                "station": station,
                "period_start": start_string,
                "period_end": end_string,
                "products": hls_products,
                "valid_pixels": valid,
                "total_pixels": total,
                "coverage_pct": coverage,
            })


coverage_table = pd.DataFrame(rows)


print("\nCOVERAGE SUMMARY\n")

print(
    coverage_table
    .groupby("source")["coverage_pct"]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.80,
            0.90,
            0.95,
            0.99,
        ]
    )
)


print("\nOBSERVATIONS ABOVE THRESHOLDS\n")

for threshold in [
    50,
    60,
    70,
    80,
    90,
    95,
    99,
]:

    summary = (
        coverage_table
        .assign(
            accepted=lambda x:
                x["coverage_pct"] >= threshold
        )
        .groupby("source")["accepted"]
        .agg(["sum", "count"])
    )

    print(
        f"\nCoverage >= {threshold}%"
    )

    print(summary)

c:\Users\usrlabsis22\AppData\Local\miniforge3\envs\et-fundacion\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


KeyboardInterrupt: 